# Metodología 3 — LLM (rúbrica)

Donde el diccionario cuenta palabras y los embeddings miden distancia semántica,
este método le pide a un LLM **leer** cada chunk y calificarlo de −2 (certeza) a +2
(incertidumbre) con una rúbrica. Responde en JSON con cuatro campos:
`relevant`, `uncertainty_score`, `confidence`, `evidence`.

> **Este notebook no llama a ningún API y no cuesta nada.** Lee las calificaciones
> que ya están en `data/processed/banxico/llm_scores/`. El paso pagado es
> `scripts/score_chunks_openrouter.py` (ver el README).

Las calificaciones vienen de un diseño 2 × 2: dos prompts (`calibrated`, `lax`) × dos modelos
(GPT-6.1 Sol y GPT-6 Luna, vía OpenRouter). La versión principal, **calibrated + Sol**, se eligió
antes de ver resultados.

Escribe `banxico_{anuncios,minutas}_index_llm.parquet`: un renglón por documento, una columna por versión.

In [1]:
import sys
from pathlib import Path

import pandas as pd

# El código vive en src/, agrupado en corpus/ index/ analysis/ (sin __init__.py):
# hay que agregar cada subcarpeta a la ruta de Python. Ver src/paths.py.
sys.path.insert(0, str(Path.cwd().parent / "src"))
import paths
paths.add_src_subfolders_to_path()
import re

from llm_prompts import PROMPTS
from common import aggregate_llm_scores, standardize_to_index

In [2]:
# Los dos corpus. Cada uno produce su propio índice: un valor por documento.
CORPORA = ["banxico_anuncios", "banxico_minutas"]

## La rúbrica

**El prompt *es* la metodología.** Cambiarlo cambia lo que mide el índice. Todos
los prompts viven en `src/index/llm_prompts.py`, y cada calificación guarda el hash del
prompt con el que se hizo, así que dos corridas con prompts distintos nunca se
mezclan sin que se note. Este es el prompt principal de los anuncios:

In [3]:
print(PROMPTS["banxico_anuncios_calibrated"])

You are an expert analyst of Banco de México's monetary-policy communication.
You classify short excerpts (chunks) from Banxico's monetary policy
announcements (comunicados de política monetaria), the statement published on
the day of each policy decision, for a Monetary Policy Uncertainty (MPU) index.

YOUR TASK
Measure how much uncertainty about the macroeconomic and monetary-policy
outlook the excerpt conveys: inflation, the policy/interest-rate path, growth,
the exchange rate, external conditions, and the balance of risks.

CALIBRATE TO BANXICO'S NORMAL LANGUAGE
Banxico's statements discuss risks in almost every paragraph. A list of the
risks to inflation and growth, and a sentence on whether the balance of risks
is biased up or down, appear in nearly every statement, whether conditions are
calm or turbulent. That routine language is the baseline and scores 0.
Score above 0 only when the excerpt conveys MORE uncertainty than a typical
Banxico statement: uncertainty that is named, e

## Las corridas en disco

In [4]:
def cargar_corridas(corpus):
    # Una corrida completa es {corpus}__{variante}__{modelo}.parquet
    runs = {}
    for path in sorted((paths.PROCESSED_DIR / "llm_scores").glob(f"{corpus}__*.parquet")):
        parts = path.stem.split("__")
        if len(parts) != 3:
            continue   # pilotos o repeticiones (no incluidos)
        _, variant, model = parts
        model = re.sub(r"[^0-9a-z]+", "_", model.split("_", 1)[1])  # openai_gpt-6.1-sol -> gpt_6_1_sol
        runs[f"llm_{variant}_{model}"] = pd.read_parquet(path)
    return runs

RUNS = {corpus: cargar_corridas(corpus) for corpus in CORPORA}

for corpus, runs in RUNS.items():
    print(corpus)
    for run, scores in runs.items():
        n_failed = int(scores["relevant"].isna().sum())  # la llamada falló: no hay veredicto
        prompts = scores["prompt_sha256"].str[:12].unique()
        print(f"  {run:32s} chunks {len(scores):5d}  sin veredicto {n_failed:3d}  "
              f"relevantes {int((scores['relevant'] == True).sum()):5d}  prompt {', '.join(prompts)}")  # noqa: E712

banxico_anuncios
  llm_calibrated_gpt_6_luna        chunks   578  sin veredicto   0  relevantes   578  prompt afe54faf91b0
  llm_calibrated_gpt_6_1_sol       chunks   578  sin veredicto   0  relevantes   577  prompt afe54faf91b0
  llm_lax_gpt_6_luna               chunks   578  sin veredicto   0  relevantes   578  prompt 2c7a50951b61
  llm_lax_gpt_6_1_sol              chunks   578  sin veredicto   0  relevantes   577  prompt 2c7a50951b61
banxico_minutas
  llm_calibrated_gpt_6_luna        chunks  2113  sin veredicto   0  relevantes  2113  prompt 0cc58068b29d
  llm_calibrated_gpt_6_1_sol       chunks  2113  sin veredicto   0  relevantes  2113  prompt 0cc58068b29d
  llm_lax_gpt_6_luna               chunks  2113  sin veredicto   0  relevantes  2113  prompt 363e0bc08023
  llm_lax_gpt_6_1_sol              chunks  2113  sin veredicto   0  relevantes  2113  prompt 363e0bc08023


## Distribución de las calificaciones

Qué tanto usa cada corrida cada punto de la escala. Una corrida que pone casi todo
en +1 y +2 deja poco espacio para que el índice se mueva entre documentos; por eso
el prompt `calibrated` le dice al modelo que el lenguaje rutinario de riesgos de
Banxico vale 0.

In [5]:
for corpus, runs in RUNS.items():
    print(corpus)
    display(pd.DataFrame({run: scores["uncertainty_score"].value_counts(normalize=True)
                          for run, scores in runs.items()}).sort_index().round(2))

banxico_anuncios


,llm_calibrated_gpt_6_luna,llm_calibrated_gpt_6_1_sol,llm_lax_gpt_6_luna,llm_lax_gpt_6_1_sol
uncertainty_score,,,,
-2.0,NaN,NaN,0.00,0.00
-1.0,0.04,0.11,0.07,0.07
0.0,0.22,0.32,0.11,0.10
1.0,0.59,0.49,0.54,0.53
2.0,0.16,0.09,0.29,0.29


banxico_minutas


,llm_calibrated_gpt_6_luna,llm_calibrated_gpt_6_1_sol,llm_lax_gpt_6_luna,llm_lax_gpt_6_1_sol
uncertainty_score,,,,
-2.0,NaN,0.00,NaN,0.00
-1.0,0.00,0.02,0.03,0.03
0.0,0.07,0.19,0.05,0.08
1.0,0.59,0.62,0.49,0.50
2.0,0.34,0.16,0.43,0.38


## Agregación

El índice del documento es el promedio de `uncertainty_score` sobre los chunks
que el LLM marcó relevantes. A diferencia de las otras dos metodologías, aquí el
filtro de relevancia viene del propio modelo: la rúbrica le dice que deje el
puntaje vacío si el chunk no trata del panorama macro/monetario.

In [6]:
for corpus, runs in RUNS.items():
    documents = pd.read_parquet(paths.PROCESSED_DIR / f"{corpus}_documents.parquet")
    # Las minutas se fechan por la decisión que discuten, no por su publicación.
    date_columns = [c for c in ["publication_date", "decision_date"] if c in documents]

    llm_index = None
    for run, scores in runs.items():
        column = f"mpu_{run}"
        run_index = aggregate_llm_scores(scores, output_column=column)
        run_index[f"idx_{column}"] = standardize_to_index(run_index[column])
        llm_index = run_index if llm_index is None else llm_index.merge(run_index, on="document_id", how="outer")

    # Fechas y títulos, para que el archivo sea legible por sí solo.
    llm_index = llm_index.merge(documents[["document_id", *date_columns, "title"]], on="document_id")
    llm_index = llm_index.sort_values("publication_date").reset_index(drop=True)
    llm_index.to_parquet(paths.PROCESSED_DIR / f"{corpus}_index_llm.parquet", index=False)
    print(f"{corpus}: guardado, {len(llm_index)} documentos")
llm_index.head()

banxico_anuncios: guardado, 238 documentos
banxico_minutas: guardado, 126 documentos


,document_id,mpu_llm_calibrated_gpt_6_luna,idx_mpu_llm_calibrated_gpt_6_luna,mpu_llm_calibrated_gpt_6_1_sol,idx_mpu_llm_calibrated_gpt_6_1_sol,mpu_llm_lax_gpt_6_luna,idx_mpu_llm_lax_gpt_6_luna,mpu_llm_lax_gpt_6_1_sol,idx_mpu_llm_lax_gpt_6_1_sol,publication_date,decision_date,title
0,banxico_min_2011_01_21,1.142857,94.630185,1.000000,102.329787,1.142857,92.111138,1.000000,90.037490,2011-02-04,2011-01-21,Minuta de la reunión de la Junta de Gobierno d...
1,banxico_min_2011_03_04,1.300000,101.754479,1.100000,106.557822,1.400000,102.592667,1.500000,110.356427,2011-03-18,2011-03-04,Minuta de la reunión de la Junta de Gobierno d...
2,banxico_min_2011_04_15,1.200000,97.220837,0.800000,93.873716,1.300000,98.516517,1.300000,102.228852,2011-04-29,2011-04-15,Minuta de la reunión de la Junta de Gobierno d...
3,banxico_min_2011_05_27,1.000000,88.153555,0.846154,95.825117,1.230769,95.694567,1.153846,96.289471,2011-06-10,2011-05-27,Minuta de la reunión de la Junta de Gobierno d...
4,banxico_min_2011_07_08,1.300000,101.754479,1.100000,106.557822,1.300000,98.516517,1.300000,102.228852,2011-07-22,2011-07-08,Minuta de la reunión de la Junta de Gobierno d...
